# Logistic Regression - Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import wandb
import dotenv

from src.api.run import sweep_logistic_regression
from src.api.sweep import wandb_sweep

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [3]:
max_runs = 100
sweep_config = {
    "name": "Logistic Regression",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "logistic_regression_config": {
            "parameters": {
                "penalty": {"values": ["l1", "l2", "elasticnet", None]},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "solver": {"values": ["lbfgs", "liblinear", "saga"]},
                "tol": {"min": 1e-5, "max": 1e-3},
                "max_iter": {"value": 1000},
                "l1_ratio": {"distribution": "uniform", "min": 0.0, "max": 1.0},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_logistic_regression, run_count=max_runs, project="logistic-regression")

## Submission from Best Model

In [5]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.logistic_regression import LogisticRegressionModel, LogisticRegressionHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [6]:
# Replace with your best run ID from W&B
run = wandb.Api().run("j7xmtprd")
config = run.config
config

{'run_config': {'num_features': 5, 'start_season': 2003, 'valid_season': 2024},
 'logistic_regression_config': {'C': 34.53696852319429,
  'tol': 0.0005380404159075536,
  'solver': 'saga',
  'penalty': None,
  'l1_ratio': 0.5184831647240584,
  'max_iter': 1000}}

In [7]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=5, valid_season=2024, start_season=2003, data_loader='season_average')

In [8]:
hyperparameters = LogisticRegressionHyperparamConfig(**config.get("logistic_regression_config", {}))
hyperparameters

LogisticRegressionHyperparamConfig(penalty=None, dual=False, tol=0.0005380404159075536, C=34.53696852319429, fit_intercept=True, intercept_scaling=1.0, class_weight=None, random_state=42, solver='saga', max_iter=1000, verbose=0, warm_start=False, n_jobs=-1, l1_ratio=0.5184831647240584)

In [9]:
model = LogisticRegressionModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [10]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_logistic_regression_{season}.csv", fit=True)

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


metrics: {'train_brier': np.float64(0.16674966229063745)}, step: None



WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_logistic_regression_2025.csv')